In [0]:
# --- 1. SET CONTEXT ---
spark.sql("USE CATALOG healthcare_p360")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

from pyspark.sql.functions import col, count, desc

# Load the Silver record we just made
df_silver = spark.table("silver.unified_patient_health_record")

# --- 2. CALCULATE HIGH-RISK PATIENTS ---
# We want a list of patients who have abnormal lab flags
df_high_risk = df_silver.filter(col("is_abnormal") == True) \
    .groupBy("patient_id", "first_name", "last_name") \
    .agg(count("is_abnormal").alias("total_abnormal_tests"))

# --- 3. CALCULATE COMMON DIAGNOSES ---
df_diagnosis_stats = df_silver.groupBy("diagnosis_desc") \
    .agg(count("patient_id").alias("patient_count")) \
    .orderBy(desc("patient_count"))

# --- 4. SAVE TO GOLD ---
df_high_risk.write.mode("overwrite").saveAsTable("gold.high_risk_patients")
df_diagnosis_stats.write.mode("overwrite").saveAsTable("gold.top_diagnoses")

print("✅ Gold Layer complete! Analytics tables are ready for reporting.")